# EXP-014: Logit-space Blending (post-hoc, no retraining)

**근거:** EXP-002~013에서 모든 blending은 **linear weighted average** 사용. 외부 노트북 `diversity-correction-pipeline`에서 본 기법 자체 적용:
```
blended = sigmoid((1-w)·logit(anchor) + w·logit(support))
```

**원리**: AUC는 ranking metric. logit space는 unbounded라 linear meta-model이 자연스럽게 합성. 0/1 영역 근처에선 logit이 큰 절대값 가져 모델 간 disagreement가 sharp하게 반영됨.

**비교 대상**:
- EXP-013 linear blend opt: OOF 0.95063, LB 0.94986
- EXP-013 linear blend equal: OOF 0.95057
- **EXP-014 logit blend opt**: OOF ?, LB ?

**비용**: 학습 없이 EXP-013 OOF/test predictions 재활용 → **5분 안**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Load EXP-013 OOF (saved as .npy)
oof_xgb = np.load('../submissions/exp013_xgb_oof.npy')
oof_lgb = np.load('../submissions/exp013_lgb_oof.npy')
oof_cat = np.load('../submissions/exp013_cat_oof.npy')
oof_mlp = np.load('../submissions/exp013_plr_mlp_ens_oof.npy')

# Load EXP-013 test predictions from submission CSVs
test_xgb = pd.read_csv('../submissions/submission_exp013_xgb.csv')['PitNextLap'].values
test_lgb = pd.read_csv('../submissions/submission_exp013_lgb.csv')['PitNextLap'].values
test_cat = pd.read_csv('../submissions/submission_exp013_cat.csv')['PitNextLap'].values
test_mlp = pd.read_csv('../submissions/submission_exp013_plr_mlp_ens.csv')['PitNextLap'].values

# Original labels
train = pd.read_csv('../data/train.csv')
test_ids = pd.read_csv('../data/test.csv')['id']
y = train['PitNextLap'].astype(int).values

# Sanity
print(f'OOF shapes: XGB {oof_xgb.shape}, LGB {oof_lgb.shape}, CAT {oof_cat.shape}, MLP {oof_mlp.shape}')
print(f'Test shapes: XGB {test_xgb.shape}, LGB {test_lgb.shape}, CAT {test_cat.shape}, MLP {test_mlp.shape}')
print(f'OOF single-model AUC:')
print(f'  XGB: {roc_auc_score(y, oof_xgb):.5f}')
print(f'  LGB: {roc_auc_score(y, oof_lgb):.5f}')
print(f'  CAT: {roc_auc_score(y, oof_cat):.5f}')
print(f'  MLP: {roc_auc_score(y, oof_mlp):.5f}')

OOF shapes: XGB (439140,), LGB (439140,), CAT (439140,), MLP (439140,)
Test shapes: XGB (188165,), LGB (188165,), CAT (188165,), MLP (188165,)
OOF single-model AUC:
  XGB: 0.94946
  LGB: 0.94933
  CAT: 0.94956
  MLP: 0.94763


## 1. Logit / Sigmoid 헬퍼

In [2]:
EPS = 1e-7
def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# 변환 후 logit OOF
logit_xgb = logit(oof_xgb); logit_lgb = logit(oof_lgb); logit_cat = logit(oof_cat); logit_mlp = logit(oof_mlp)
logit_test_xgb = logit(test_xgb); logit_test_lgb = logit(test_lgb); logit_test_cat = logit(test_cat); logit_test_mlp = logit(test_mlp)

print(f'logit OOF stats:')
for name, l in [('XGB', logit_xgb), ('LGB', logit_lgb), ('CAT', logit_cat), ('MLP', logit_mlp)]:
    print(f'  {name}: mean {l.mean():.3f}, std {l.std():.3f}, min {l.min():.3f}, max {l.max():.3f}')

logit OOF stats:
  XGB: mean -3.267, std 2.983, min -10.332, max 5.579
  LGB: mean -3.238, std 2.940, min -10.345, max 5.365
  CAT: mean -3.215, std 2.962, min -13.026, max 6.581
  MLP: mean -3.590, std 3.358, min -15.130, max 4.479


## 2. Linear vs Logit Blending — Equal

In [3]:
# Linear equal
oof_lin_eq  = (oof_xgb + oof_lgb + oof_cat + oof_mlp) / 4
test_lin_eq = (test_xgb + test_lgb + test_cat + test_mlp) / 4
auc_lin_eq  = roc_auc_score(y, oof_lin_eq)

# Logit equal
oof_logit_eq  = sigmoid((logit_xgb + logit_lgb + logit_cat + logit_mlp) / 4)
test_logit_eq = sigmoid((logit_test_xgb + logit_test_lgb + logit_test_cat + logit_test_mlp) / 4)
auc_logit_eq  = roc_auc_score(y, oof_logit_eq)

print(f'Equal blend OOF AUC comparison:')
print(f'  Linear: {auc_lin_eq:.5f}  (EXP-013 record: 0.95057)')
print(f'  Logit : {auc_logit_eq:.5f}  Δ = {auc_logit_eq - auc_lin_eq:+.5f}')

Equal blend OOF AUC comparison:
  Linear: 0.95057  (EXP-013 record: 0.95057)
  Logit : 0.95051  Δ = -0.00006


## 3. Linear vs Logit Blending — Optimal grid search

In [4]:
# Linear opt (sanity check, EXP-013 재현)
best_lin = (None, -1)
for w_xgb in np.arange(0, 1.001, 0.05):
    for w_lgb in np.arange(0, 1.001 - w_xgb, 0.05):
        for w_cat in np.arange(0, 1.001 - w_xgb - w_lgb, 0.05):
            w_mlp = 1 - w_xgb - w_lgb - w_cat
            if w_mlp < 0: continue
            s = w_xgb*oof_xgb + w_lgb*oof_lgb + w_cat*oof_cat + w_mlp*oof_mlp
            a = roc_auc_score(y, s)
            if a > best_lin[1]:
                best_lin = ((round(w_xgb,2), round(w_lgb,2), round(w_cat,2), round(w_mlp,2)), a)

# Logit opt
best_logit = (None, -1)
for w_xgb in np.arange(0, 1.001, 0.05):
    for w_lgb in np.arange(0, 1.001 - w_xgb, 0.05):
        for w_cat in np.arange(0, 1.001 - w_xgb - w_lgb, 0.05):
            w_mlp = 1 - w_xgb - w_lgb - w_cat
            if w_mlp < 0: continue
            z = w_xgb*logit_xgb + w_lgb*logit_lgb + w_cat*logit_cat + w_mlp*logit_mlp
            s = sigmoid(z)
            a = roc_auc_score(y, s)
            if a > best_logit[1]:
                best_logit = ((round(w_xgb,2), round(w_lgb,2), round(w_cat,2), round(w_mlp,2)), a)

w_lin, auc_lin_opt = best_lin
w_logit, auc_logit_opt = best_logit

oof_lin_opt  = w_lin[0]*oof_xgb + w_lin[1]*oof_lgb + w_lin[2]*oof_cat + w_lin[3]*oof_mlp
test_lin_opt = w_lin[0]*test_xgb + w_lin[1]*test_lgb + w_lin[2]*test_cat + w_lin[3]*test_mlp

z_logit_opt = w_logit[0]*logit_xgb + w_logit[1]*logit_lgb + w_logit[2]*logit_cat + w_logit[3]*logit_mlp
oof_logit_opt = sigmoid(z_logit_opt)
z_logit_test_opt = w_logit[0]*logit_test_xgb + w_logit[1]*logit_test_lgb + w_logit[2]*logit_test_cat + w_logit[3]*logit_test_mlp
test_logit_opt = sigmoid(z_logit_test_opt)

print('=== EXP-014 결과 (post-hoc, EXP-013 OOF 재사용) ===')
print(f'Linear opt {w_lin}: {auc_lin_opt:.5f}  (EXP-013 record: 0.95063)')
print(f'Logit  opt {w_logit}: {auc_logit_opt:.5f}  Δ = {auc_logit_opt - auc_lin_opt:+.5f}')
print()
if auc_logit_opt > auc_lin_opt + 0.00001:
    print(f'  → Logit blending OOF +{auc_logit_opt - auc_lin_opt:.5f} 개선. LB 측정 가치 있음.')
elif auc_logit_opt > auc_lin_opt - 0.00001:
    print(f'  → Linear/Logit 거의 동일. LB 차이 미미 예상.')
else:
    print(f'  → Logit OOF {auc_logit_opt - auc_lin_opt:+.5f} 손실. Linear가 더 나음.')

=== EXP-014 결과 (post-hoc, EXP-013 OOF 재사용) ===
Linear opt (np.float64(0.2), np.float64(0.2), np.float64(0.4), np.float64(0.2)): 0.95063  (EXP-013 record: 0.95063)
Logit  opt (np.float64(0.2), np.float64(0.2), np.float64(0.4), np.float64(0.2)): 0.95057  Δ = -0.00006

  → Logit OOF -0.00006 손실. Linear가 더 나음.


## 4. Submissions

In [5]:
for name, prob in [
    ('exp014_logit_equal', test_logit_eq),
    ('exp014_logit_opt',   test_logit_opt),
    ('exp014_linear_equal', test_lin_eq),  # control
    ('exp014_linear_opt',   test_lin_opt), # control (= EXP-013)
]:
    sub = pd.DataFrame({'id': test_ids, 'PitNextLap': prob})
    path = f'../submissions/submission_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'  saved {path}  mean={prob.mean():.4f}')

print()
print(f'Linear opt weights : {w_lin}')
print(f'Logit  opt weights : {w_logit}')

  saved ../submissions/submission_exp014_logit_equal.csv  mean=0.1979
  saved ../submissions/submission_exp014_logit_opt.csv  mean=0.1978
  saved ../submissions/submission_exp014_linear_equal.csv  mean=0.1984
  saved ../submissions/submission_exp014_linear_opt.csv  mean=0.1983

Linear opt weights : (np.float64(0.2), np.float64(0.2), np.float64(0.4), np.float64(0.2))
Logit  opt weights : (np.float64(0.2), np.float64(0.2), np.float64(0.4), np.float64(0.2))
